In [1]:
import pandas as pd
import numpy as np

from sklearn.experimental import enable_iterative_imputer #A
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor

In [2]:
df_matches = pd.read_csv('../../data/preprocessed/preprocessed_1.csv')
df_matches.sort_values(by=["season", "stage", "date"], inplace=True)
season_labels, _ = pd.factorize(df_matches['result_match'])
feature_cols = [col for col in df_matches.columns if col not in
                ["result_match", "stage", "season" ,"date", "home_team", "away_team"]]

df = df_matches[["match_api_id", "result_match", "stage", "season", "date", "home_team", "away_team"]]

df_matches = df_matches[feature_cols]
df_matches

,match_api_id,points_home,points_away,home_last_team_goal,home_last_team_shoton,home_last_team_possession,away_last_team_goal,away_last_team_shoton,away_last_team_possession,team_strength_home,...,rolling_avg_shoton_diff,rolling_stability_shoton_diff,points_diff,rolling_avg_goals_ratio,rolling_stability_goal_ratio,rolling_avg_goals_conversion_rate_ratio,rolling_stability_goals_conversion_rate_ratio,rolling_avg_shoton_ratio,rolling_stability_shoton_ratio,points_ratio
0,489063,3,4,1.00,12.0,34.0,1.000000,1.0,55.0,78.318182,...,11.00,NaN,-1,1.000000,NaN,0.083333,NaN,12.000000,NaN,0.750000
1,489065,3,4,2.00,5.0,48.0,1.000000,8.0,51.0,65.136364,...,-3.00,NaN,-1,2.000000,NaN,3.200000,NaN,0.625000,NaN,0.750000
2,489068,3,3,1.00,7.0,47.0,3.000000,1.0,47.0,63.924242,...,6.00,NaN,0,0.333333,NaN,0.047619,NaN,7.000000,NaN,1.000000
3,489069,3,0,3.00,5.0,53.0,1.000000,2.0,66.0,71.318182,...,3.00,NaN,3,3.000000,NaN,1.200000,NaN,2.500000,NaN,0.000000
4,489070,4,0,1.00,5.0,47.0,1.000000,7.0,52.0,63.318182,...,-2.00,NaN,4,1.000000,NaN,1.400000,NaN,0.714286,NaN,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2973,1987599,44,34,1.00,4.0,44.0,4.000000,6.0,69.0,72.327273,...,-1.00,1.933017e-01,10,0.387097,0.436436,0.998225,4.603260,0.838710,1.077632,1.294118
2974,1987598,49,80,2.00,5.0,53.0,3.000000,4.0,67.0,65.445455,...,-1.25,-5.252551e-01,-31,1.027778,0.762362,1.098123,0.211186,0.800000,0.699854,0.612500
2975,1987597,68,17,2.00,7.0,31.0,1.666667,1.0,65.0,70.627273,...,3.00,4.440892e-16,51,1.239130,1.769303,0.399248,0.159060,1.555556,1.000000,4.000000
2976,1987601,34,70,1.75,6.0,50.0,1.000000,6.0,47.0,79.690909,...,-2.80,1.407746e+00,-36,0.795455,0.635934,2.092939,2.519644,0.562500,2.573908,0.485714


In [3]:
(
    df_matches.isna()
    .sum(axis=0)
)

match_api_id                                      0
points_home                                       0
points_away                                       0
home_last_team_goal                               0
home_last_team_shoton                             0
home_last_team_possession                         0
away_last_team_goal                               0
away_last_team_shoton                             0
away_last_team_possession                         0
team_strength_home                                0
team_strength_away                                0
strength_difference                               0
team_aggression_home                              0
team_aggression_away                              0
aggression_difference                             0
team_acceleration_home                            0
team_acceleration_away                            0
acceleration_difference                           0
goal_conversion_rate_home                         0
goal_convers

In [4]:
simple_imputer = SimpleImputer()
Xm_si = simple_imputer.fit_transform(df_matches)

rf = RandomForestRegressor(random_state=0, n_jobs=-1)
multivariate_imputer = IterativeImputer(estimator=rf, max_iter=10, tol=0.01)
Xm_ii = multivariate_imputer.fit_transform(df_matches)

imputed_si_df = pd.DataFrame(Xm_si, columns=df_matches.columns)
imputed_ii_df = pd.DataFrame(Xm_ii, columns=df_matches.columns)

In [5]:
(imputed_si_df.isna().sum(axis=0))

match_api_id                                     0
points_home                                      0
points_away                                      0
home_last_team_goal                              0
home_last_team_shoton                            0
home_last_team_possession                        0
away_last_team_goal                              0
away_last_team_shoton                            0
away_last_team_possession                        0
team_strength_home                               0
team_strength_away                               0
strength_difference                              0
team_aggression_home                             0
team_aggression_away                             0
aggression_difference                            0
team_acceleration_home                           0
team_acceleration_away                           0
acceleration_difference                          0
goal_conversion_rate_home                        0
goal_conversion_rate_away      

In [6]:
(imputed_ii_df.isna().sum(axis=0))

match_api_id                                     0
points_home                                      0
points_away                                      0
home_last_team_goal                              0
home_last_team_shoton                            0
home_last_team_possession                        0
away_last_team_goal                              0
away_last_team_shoton                            0
away_last_team_possession                        0
team_strength_home                               0
team_strength_away                               0
strength_difference                              0
team_aggression_home                             0
team_aggression_away                             0
aggression_difference                            0
team_acceleration_home                           0
team_acceleration_away                           0
acceleration_difference                          0
goal_conversion_rate_home                        0
goal_conversion_rate_away      

In [7]:
imputed_ii_df=pd.merge(df, imputed_ii_df, on="match_api_id")

In [9]:
imputed_ii_df.to_csv('../../data/preprocessed/imputed_data.csv', index=False)